# Problem 4 - Đánh giá Năng lực Thúc đẩy Doanh số và Thái độ Phục vụ Khách hàng của Nhân viên

**Mục tiêu:** Cung cấp dữ liệu minh bạch cho bộ phận Nhân sự nhằm tính toán KPI, chính sách thưởng/phạt, đồng thời đánh giá thái độ phục vụ khách hàng và chất lượng giao dịch của từng nhân sự bán hàng (`sales_employee_id`).

**Nguồn dữ liệu (silver, đã chuẩn hóa 3NF):**
- `ORDER` (order_id PK; customer_id, zip, sales_employee_id, order_date, order_status, ...)
- `ORDER_ITEMS` (order_id + product_id PK; quantity, unit_price, discount_amount)
- `REVIEWS` (order_id / review thông qua liên kết đơn hàng; rating, review_title)
- `RETURNS` (order_id; return_reason)

**Framework áp dụng:** Measure → Metric → KPI
| Bước | Nội dung |
|---|---|
| Measure | `quantity`, `unit_price`, `discount_amount` (ORDER_ITEMS); `order_status` (ORDER); `rating` (REVIEWS); `return_reason` (RETURNS) |
| Metric | `SUM(doanh_thu_dòng)`, `COUNT(DISTINCT order_id)`, `AVG(rating)`, `COUNT(return_id)` theo `sales_employee_id` |
| KPI | Tổng doanh số, Số đơn thành công, Doanh thu TB/hóa đơn, Điểm đánh giá TB (Rating), Tỷ lệ đánh giá thấp, Tỷ lệ hoàn trả đơn hàng |


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["figure.figsize"] = (12, 6)


## 1. Load dữ liệu (silver layer)

In [2]:
ORDER_PATH = "DAAI_N1.4/silver_data/ORDER.csv"
ORDER_ITEMS_PATH = "DAAI_N1.4/silver_data/ORDER_ITEMS.csv"
REVIEWS_PATH = "DAAI_N1.4/silver_data/REVIEWS.csv"
RETURNS_PATH = "DAAI_N1.4/silver_data/RETURNS.csv"

df_order = pd.read_csv(ORDER_PATH, parse_dates=["order_date"])
df_items = pd.read_csv(ORDER_ITEMS_PATH)
df_reviews = pd.read_csv(REVIEWS_PATH) if pd.io.common.file_exists(REVIEWS_PATH) else None
df_returns = pd.read_csv(RETURNS_PATH) if pd.io.common.file_exists(RETURNS_PATH) else None

print("ORDER:", df_order.shape)
print("ORDER_ITEMS:", df_items.shape)


## 2. Tính toán Doanh số và Trạng thái Đơn hàng

In [3]:
df_items["doanh_thu_dong"] = (
    df_items["quantity"] * df_items["unit_price"] - df_items["discount_amount"]
)

# Tổng hợp doanh thu theo từng order_id
order_revenue = df_items.groupby("order_id")["doanh_thu_dong"].sum().reset_index()
df_order_full = df_order.merge(order_revenue, on="order_id", how="left")


## 3. Tích hợp Đánh giá (Reviews) và Hoàn trả (Returns) theo Nhân viên

In [4]:
# Gắn thông tin sales_employee_id vào reviews và returns thông qua order_id
if df_reviews is not None and "sales_employee_id" not in df_reviews.columns:
    df_reviews = df_reviews.merge(df_order[["order_id", "sales_employee_id"]], on="order_id", how="inner")

if df_returns is not None and "sales_employee_id" not in df_returns.columns:
    df_returns = df_returns.merge(df_order[["order_id", "sales_employee_id"]], on="order_id", how="inner")


## 4. Xây dựng Tổng hợp KPI theo Nhân viên (SalesPersonID)

In [5]:
# Metric cơ bản từ đơn hàng
metric_sales = df_order_full.groupby("sales_employee_id").agg(
    tong_so_hoa_don=("order_id", "count"),
    so_hoa_don_thanh_cong=("order_status", lambda x: (x == "delivered").sum()),
    tong_doanh_so=("doanh_thu_dong", "sum"),
    doanh_thu_tb_hoa_don=("doanh_thu_dong", "mean")
).reset_index()

# Metric đánh giá thái độ phục vụ (nếu có bảng reviews)
if df_reviews is not None and "rating" in df_reviews.columns:
    metric_reviews = df_reviews.groupby("sales_employee_id").agg(
        diem_rating_tb=("rating", "mean"),
        so_luong_danh_gia=("rating", "count"),
        ty_le_danh_gia_thap=("rating", lambda x: (x <= 2).mean() * 100)
    ).reset_index()
    metric_sales = metric_sales.merge(metric_reviews, on="sales_employee_id", how="left")

# Metric tỷ lệ hoàn trả đơn hàng (nếu có bảng returns)
if df_returns is not None:
    metric_returns = df_returns.groupby("sales_employee_id").agg(
        so_luong_hoan_tra=(
            "order_id" if "order_id" in df_returns.columns else df_returns.columns[0],
            "count"
        )
    ).reset_index()
    metric_sales = metric_sales.merge(metric_returns, on="sales_employee_id", how="left")
    metric_sales["so_luong_hoan_tra"] = metric_sales["so_luong_hoan_tra"].fillna(0)
    metric_sales["ty_le_hoan_tra"] = (metric_sales["so_luong_hoan_tra"] / metric_sales["tong_so_hoa_don"]) * 100

# Tỷ lệ chuyển đổi thành công
metric_sales["ty_le_chuyen_doi"] = (metric_sales["so_hoa_don_thanh_cong"] / metric_sales["tong_so_hoa_don"]) * 100

kpi_table = metric_sales.sort_values(by="tong_doanh_so", ascending=False).reset_index(drop=True)
kpi_table.head(10)


## 5. Xuất kết quả KPI cho Problem 4

In [6]:
OUTPUT_PATH = "DAAI_N1.4/problem_statements/output/ps4_kpi_customer_service.csv"
kpi_table.to_csv(OUTPUT_PATH, index=False)
print(f"Đã lưu bảng KPI Problem 4 vào: {OUTPUT_PATH}")
